# Adia Lab Causal Discovery — 1st Place Solution

Full implementation of the top solution described at [thetourney.github.io/adia-report](https://thetourney.github.io/adia-report/).

### Architecture
- **Preprocessing**: Edge tensors with 3 channels (sorted u, sorted v, kernel regression coefficient)
- **Stem layer**: 3→64 channels linear
- **5× Residual Conv1D blocks**: GroupNorm + GELU + residual
- **Average pooling**: → 64-dim edge embeddings
- **Merge operator**: edge emb + edge-type embedding → LayerNorm + GELU
- **2× Self-attention layers**
- **Edge head** (training only): binary adjacency matrix
- **Node head**: gather 4 edges (u→X, u→Y, X→u, Y→u) → 8-class classification
- **Training**: AdamW + Cosine Annealing, inverse-frequency class weights, dual CE loss

### Setup

In [ ]:
# %pip install pytorch_lightning
# %pip install crunch-cli --upgrade --quiet

### Imports & Setup

In [ ]:
import typing
import os
from tqdm.auto import tqdm

import pandas as pd
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import pytorch_lightning as pl

import networkx as nx
from sklearn.model_selection import train_test_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
D_MODEL = 64
N_OBS = 1000
MAX_VARS = 10
N_CLASSES = 8   # node classes
N_EDGE_TYPES = 7  # see preprocessing

CLASS_NAMES = [
    "Confounder", "Collider", "Mediator",
    "Cause of X", "Cause of Y",
    "Consequence of X", "Consequence of Y",
    "Independent",
]

### Data Preprocessing

In [ ]:
def compute_kernel_regression_coefficients(
    data: np.ndarray,  # shape (N, p)
    target_col: int,
    bandwidth: float = 0.5,
) -> np.ndarray:
    """
    Multivariate kernel regression for node j given all other nodes.
    Returns coefficients c of shape (N, p) where c[:, target_col]=0 (intercept
    stored at index 0 conceptually) – here we return the full coefficient vector
    per observation (size p) by solving a weighted local linear regression.
    
    For observation i, minimize:
        sum_l  w(u_i, u_l) * (u_l[j] - c_i0 - sum_{k!=j} c_ik * u_l[k])^2
    where w is a Gaussian kernel on the full row distance.
    """
    N, p = data.shape
    # pairwise squared distances in full variable space
    # shape (N, N)
    diff = data[:, None, :] - data[None, :, :]   # (N, N, p)
    sq_dist = (diff ** 2).sum(axis=-1)            # (N, N)
    weights = np.exp(-sq_dist / (2 * bandwidth ** 2))  # (N, N)

    # Build design matrix excluding target column
    other_cols = [k for k in range(p) if k != target_col]
    X_design = np.concatenate(
        [np.ones((N, 1)), data[:, other_cols]], axis=1
    )  # (N, p)  — intercept + (p-1) predictors
    y_target = data[:, target_col]  # (N,)

    # Solve for each observation i: (X^T W_i X) c_i = X^T W_i y
    # Use closed-form WLS: c_i = (X^T diag(w_i) X)^{-1} X^T diag(w_i) y
    coeffs = np.zeros((N, p))  # store intercept + (p-1) predictors
    for i in range(N):
        w_i = weights[i]  # (N,)
        W = np.diag(w_i)
        A = X_design.T @ W @ X_design  # (p, p)
        b = X_design.T @ (w_i * y_target)  # (p,)
        try:
            c = np.linalg.solve(A + 1e-6 * np.eye(p), b)
        except np.linalg.LinAlgError:
            c = np.zeros(p)
        coeffs[i] = c

    return coeffs  # (N, p): [intercept, coeff_{k1}, coeff_{k2}, ...]


def build_edge_tensor(
    df: pd.DataFrame,
) -> tuple:
    """
    Given a DataFrame of shape (1000, p), build:
      - edge_data: shape (p*(p-1), 3, 1000)
          channel 0: sorted observations of source u
          channel 1: observations of target v (sorted by u)
          channel 2: kernel regression coefficient of u->v at each obs point
      - edge_types: shape (p*(p-1),) — integer 0-6
    
    Edge type encoding for u->v:
      0: u=X, v!=Y
      1: u=X, v=Y
      2: u=Y, v!=X
      3: u=Y, v=X
      4: u!=X, u!=Y, v=X
      5: u!=X, u!=Y, v=Y
      6: neither
    """
    cols = list(df.columns)
    p = len(cols)
    data = df.values.astype(np.float32)  # (1000, p)
    N = data.shape[0]

    edges = []
    edge_types = []

    for i, u_name in enumerate(cols):
        # Sort by column u for this source
        sort_idx = np.argsort(data[:, i])
        u_sorted = data[sort_idx, i]  # (N,)

        for j, v_name in enumerate(cols):
            if i == j:
                continue

            v_sorted_by_u = data[sort_idx, j]  # (N,)

            # Kernel regression coefficient channel:
            # We want the coefficient c_{i,k->j} for the edge u->v at each obs
            # For efficiency, compute a simple local linear regression coeff
            # (the u component only — a scalar per observation)
            # Full multivariate is expensive; use pairwise kernel regression
            u_all = data[:, i]  # (N,)
            v_all = data[:, j]  # (N,)
            # Pairwise Gaussian weights
            diff_u = u_all[:, None] - u_all[None, :]   # (N, N)
            w = np.exp(-diff_u ** 2 / 0.5)              # (N, N)
            # WLS: c_i = (sum_l w_il * u_l * v_l - mu_u * mu_v) / var_u
            denom = (w * (u_all[None, :] ** 2)).sum(axis=1) \
                    - ((w * u_all[None, :]).sum(axis=1)) ** 2 / w.sum(axis=1)
            numer = (w * u_all[None, :] * v_all[None, :]).sum(axis=1) \
                    - (w * u_all[None, :]).sum(axis=1) \
                    * (w * v_all[None, :]).sum(axis=1) / w.sum(axis=1)
            coeff_all = np.where(np.abs(denom) > 1e-8, numer / denom, 0.0)
            coeff_sorted = coeff_all[sort_idx]  # (N,)

            # Stack 3 channels
            edge_tensor = np.stack(
                [u_sorted, v_sorted_by_u, coeff_sorted], axis=0
            )  # (3, N)
            edges.append(edge_tensor)

            # Edge type
            u_is_X = (u_name == "X")
            u_is_Y = (u_name == "Y")
            v_is_X = (v_name == "X")
            v_is_Y = (v_name == "Y")

            if u_is_X and not v_is_Y:
                etype = 0
            elif u_is_X and v_is_Y:
                etype = 1
            elif u_is_Y and not v_is_X:
                etype = 2
            elif u_is_Y and v_is_X:
                etype = 3
            elif not u_is_X and not u_is_Y and v_is_X:
                etype = 4
            elif not u_is_X and not u_is_Y and v_is_Y:
                etype = 5
            else:
                etype = 6

            edge_types.append(etype)

    edge_data = np.stack(edges, axis=0).astype(np.float32)  # (E, 3, N)
    edge_types = np.array(edge_types, dtype=np.int64)       # (E,)
    return edge_data, edge_types


def build_node_index_map(cols: list) -> dict:
    """Return {name: index} for the column list."""
    return {name: i for i, name in enumerate(cols)}


def get_edge_index(u_idx: int, v_idx: int, p: int) -> int:
    """Convert (u, v) node indices to edge list index in the p*(p-1) list."""
    # edges are enumerated as: for u in range(p): for v in range(p): if u!=v
    count = 0
    for i in range(p):
        for j in range(p):
            if i == j:
                continue
            if i == u_idx and j == v_idx:
                return count
            count += 1
    raise ValueError(f"Edge ({u_idx},{v_idx}) not found")

### Dataset

In [ ]:
class CausalEdgeDataset(Dataset):
    """
    Each sample contains pre-computed edge data for one causal graph.
    Stores variable-length structures; collation happens in the DataLoader.
    """

    def __init__(
        self,
        X_list: typing.List[pd.DataFrame],
        y_list: typing.Optional[typing.List[pd.DataFrame]] = None,
    ):
        self.samples = []
        for idx in tqdm(range(len(X_list)), desc="Building edge dataset"):
            df = X_list[idx]
            edge_data, edge_types = build_edge_tensor(df)
            cols = list(df.columns)
            p = len(cols)

            sample = {
                "edge_data": torch.tensor(edge_data),    # (E, 3, N)
                "edge_types": torch.tensor(edge_types),  # (E,)
                "cols": cols,
                "p": p,
            }

            if y_list is not None:
                y_df = y_list[idx]
                # Adjacency matrix labels (for edge classification head)
                adj = torch.tensor(
                    y_df.values, dtype=torch.float32
                )  # (p, p)
                sample["adj"] = adj

                # Node labels (0-7) for non-X/Y nodes
                _, adjacency_label = create_graph_label()
                node_labels = {}
                for v_name in cols:
                    if v_name in ("X", "Y"):
                        continue
                    submatrix = y_df.loc[
                        [v_name, "X", "Y"], [v_name, "X", "Y"]
                    ]
                    key = tuple(submatrix.values.flatten())
                    label_str = adjacency_label.get(key, "Independent")
                    node_labels[v_name] = CLASS_NAMES.index(label_str)
                sample["node_labels"] = node_labels  # dict name->int

            self.samples.append(sample)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """
    Custom collate: each item may have different p.
    Pad edge_data to max edges in batch.
    """
    max_edges = max(item["edge_data"].shape[0] for item in batch)
    B = len(batch)
    N = batch[0]["edge_data"].shape[-1]

    edge_data_padded = torch.zeros(B, max_edges, 3, N)
    edge_types_padded = torch.zeros(B, max_edges, dtype=torch.long)
    edge_mask = torch.zeros(B, max_edges, dtype=torch.bool)

    # For adjacency targets we need to store them per-sample (variable size)
    adj_list = []
    node_labels_list = []
    cols_list = []
    p_list = []

    for b, item in enumerate(batch):
        E = item["edge_data"].shape[0]
        edge_data_padded[b, :E] = item["edge_data"]
        edge_types_padded[b, :E] = item["edge_types"]
        edge_mask[b, :E] = True
        cols_list.append(item["cols"])
        p_list.append(item["p"])

        if "adj" in item:
            adj_list.append(item["adj"])
            node_labels_list.append(item["node_labels"])

    result = {
        "edge_data": edge_data_padded,        # (B, max_E, 3, N)
        "edge_types": edge_types_padded,      # (B, max_E)
        "edge_mask": edge_mask,               # (B, max_E)
        "cols": cols_list,
        "p": p_list,
    }
    if adj_list:
        result["adj"] = adj_list
        result["node_labels"] = node_labels_list
    return result

### Model Architecture

In [ ]:
class ConvBlock(nn.Module):
    """
    Residual 1D convolutional block with GroupNorm + GELU.
    Fig. 5 in the report.
    """

    def __init__(self, d: int, kernel_size: int = 3, n_groups: int = 8):
        super().__init__()
        self.conv = nn.Conv1d(
            d, d, kernel_size, padding=kernel_size // 2, bias=False
        )
        self.norm = nn.GroupNorm(n_groups, d)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*E, d, N)
        return x + self.act(self.norm(self.conv(x)))


class MergeOperator(nn.Module):
    """
    Concatenate inputs on channel dim, reduce back to d_model.
    Fig. 6 in the report.
    """

    def __init__(self, n_inputs: int, d: int):
        super().__init__()
        self.linear = nn.Linear(n_inputs * d, d)
        self.norm = nn.LayerNorm(d)
        self.act = nn.GELU()

    def forward(self, inputs: typing.List[torch.Tensor]) -> torch.Tensor:
        # Each input: (..., d)
        x = torch.cat(inputs, dim=-1)
        return self.act(self.norm(self.linear(x)))


class StemLayer(nn.Module):
    """Linear layer expanding 3 channels -> d_model."""

    def __init__(self, d: int):
        super().__init__()
        self.linear = nn.Linear(3, d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*E, 3, N)
        # Transpose to (B*E, N, 3), apply linear, transpose back
        return self.linear(x.permute(0, 2, 1)).permute(0, 2, 1)


class EdgeFeatureExtractor(nn.Module):
    """
    Stem + 5 conv blocks + average pooling -> (B*E, d_model).
    """

    def __init__(self, d: int = D_MODEL, n_blocks: int = 5):
        super().__init__()
        self.stem = StemLayer(d)
        self.blocks = nn.Sequential(*[ConvBlock(d) for _ in range(n_blocks)])
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*E, 3, N)
        x = self.stem(x)           # (B*E, d, N)
        x = self.blocks(x)         # (B*E, d, N)
        x = self.pool(x).squeeze(-1)  # (B*E, d)
        return x


class SelfAttentionLayer(nn.Module):
    """
    Standard multi-head self-attention over edge embeddings.
    """

    def __init__(self, d: int = D_MODEL, n_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)
        )
        self.norm2 = nn.LayerNorm(d)

    def forward(
        self, x: torch.Tensor, key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        # x: (B, E, d)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class ADIAModel(nn.Module):
    """
    Full 1st-place model as described in https://thetourney.github.io/adia-report/

    Pipeline:
      1. StemLayer:           (B, E, 3, N) -> (B, E, d)   [via EdgeFeatureExtractor]
      2. Edge type embedding:  merge with (B, E, d) type embeddings
      3. 2x SelfAttentionLayer: refine edge embeddings
      4. Edge classification head (training only): (B, E, 2)
      5. Node classification head: gather 4 edges per node -> (B, num_other_nodes, 8)
    """

    def __init__(self, d: int = D_MODEL, n_edge_types: int = N_EDGE_TYPES):
        super().__init__()
        self.d = d
        self.extractor = EdgeFeatureExtractor(d)
        self.edge_type_emb = nn.Embedding(n_edge_types, d)
        self.edge_merge = MergeOperator(n_inputs=2, d=d)
        self.attn_layers = nn.ModuleList([
            SelfAttentionLayer(d) for _ in range(2)
        ])
        # Edge classification head (binary adjacency)
        self.edge_head = nn.Linear(d, 2)
        # Node classification head uses a Merge(4 edges) -> linear(8)
        self.node_merge = MergeOperator(n_inputs=4, d=d)
        self.node_head = nn.Linear(d, N_CLASSES)

    def forward(
        self,
        edge_data: torch.Tensor,    # (B, E, 3, N)
        edge_types: torch.Tensor,   # (B, E)
        edge_mask: torch.Tensor,    # (B, E) True = valid
        cols_list: typing.List[typing.List[str]],
    ) -> typing.Tuple[torch.Tensor, typing.List[torch.Tensor]]:
        """
        Returns:
          edge_logits: (B, E, 2) — for edge classification head
          node_logits_list: list of (num_other_nodes_b, 8) per sample in batch
        """
        B, E, C, N = edge_data.shape

        # --- Feature extraction ---
        x_flat = edge_data.view(B * E, C, N)                   # (B*E, 3, N)
        edge_emb = self.extractor(x_flat).view(B, E, self.d)   # (B, E, d)

        # --- Merge with edge type embeddings ---
        type_emb = self.edge_type_emb(edge_types)              # (B, E, d)
        edge_emb = self.edge_merge([edge_emb, type_emb])       # (B, E, d)

        # --- Self-attention (mask padding) ---
        # key_padding_mask: True = ignore (inverted from edge_mask)
        pad_mask = ~edge_mask  # (B, E)
        for attn in self.attn_layers:
            edge_emb = attn(edge_emb, key_padding_mask=pad_mask)

        # --- Edge classification head ---
        edge_logits = self.edge_head(edge_emb)  # (B, E, 2)

        # --- Node classification head ---
        node_logits_list = []
        for b in range(B):
            cols = cols_list[b]
            p = len(cols)
            col_idx = {name: i for i, name in enumerate(cols)}
            E_b = p * (p - 1)

            # Map (u, v) -> edge index in the flattened edge list
            edge_order = {}
            count = 0
            for ui in range(p):
                for vi in range(p):
                    if ui != vi:
                        edge_order[(ui, vi)] = count
                        count += 1

            x_idx = col_idx.get("X", None)
            y_idx = col_idx.get("Y", None)
            embs = edge_emb[b]  # (E, d)

            other_nodes = [
                name for name in cols if name not in ("X", "Y")
            ]
            if not other_nodes or x_idx is None or y_idx is None:
                node_logits_list.append(None)
                continue

            node_logits = []
            for node_name in other_nodes:
                u_idx = col_idx[node_name]
                # Gather 4 edges: u->X, u->Y, X->u, Y->u
                ux_emb = embs[edge_order[(u_idx, x_idx)]]
                uy_emb = embs[edge_order[(u_idx, y_idx)]]
                xu_emb = embs[edge_order[(x_idx, u_idx)]]
                yu_emb = embs[edge_order[(y_idx, u_idx)]]

                node_emb = self.node_merge(
                    [ux_emb, uy_emb, xu_emb, yu_emb]
                )  # (d,)
                logit = self.node_head(node_emb)  # (8,)
                node_logits.append(logit)

            if node_logits:
                node_logits_list.append(torch.stack(node_logits))  # (K, 8)
            else:
                node_logits_list.append(None)

        return edge_logits, node_logits_list

### Training (ModelWrapper)

In [ ]:
def compute_class_weights(
    y_list: typing.List[pd.DataFrame],
    adjacency_label: dict,
) -> torch.Tensor:
    """Compute inverse-frequency weights for node classification."""
    counts = torch.zeros(N_CLASSES)
    for y_df in y_list:
        cols = list(y_df.columns)
        for v_name in cols:
            if v_name in ("X", "Y"):
                continue
            submatrix = y_df.loc[[v_name, "X", "Y"], [v_name, "X", "Y"]]
            key = tuple(submatrix.values.flatten())
            label_str = adjacency_label.get(key, "Independent")
            label_idx = CLASS_NAMES.index(label_str)
            counts[label_idx] += 1
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * N_CLASSES
    return weights


class ADIAModelWrapper(pl.LightningModule):

    def __init__(
        self,
        d: int = D_MODEL,
        node_class_weights: torch.Tensor = None,
        edge_class_weights: torch.Tensor = None,
        lr: float = 1e-3,
        max_epochs: int = 20,
    ):
        super().__init__()
        self.model = ADIAModel(d)
        self.lr = lr
        self.max_epochs = max_epochs

        node_w = node_class_weights if node_class_weights is not None \
            else torch.ones(N_CLASSES)
        edge_w = edge_class_weights if edge_class_weights is not None \
            else torch.tensor([1.0, 5.0])  # upweight edges that exist

        self.node_criterion = nn.CrossEntropyLoss(weight=node_w)
        self.edge_criterion = nn.CrossEntropyLoss(weight=edge_w)

    def forward(self, batch):
        return self.model(
            batch["edge_data"].to(self.device),
            batch["edge_types"].to(self.device),
            batch["edge_mask"].to(self.device),
            batch["cols"],
        )

    def _compute_loss(self, batch, split: str):
        edge_logits, node_logits_list = self(batch)
        B = edge_logits.shape[0]
        edge_mask = batch["edge_mask"]
        cols_list = batch["cols"]
        adj_list = batch.get("adj", None)
        node_labels_list = batch.get("node_labels", None)

        total_loss = torch.tensor(0.0, device=self.device)
        n_terms = 0

        for b in range(B):
            cols = cols_list[b]
            p = len(cols)
            E_b = p * (p - 1)
            col_idx = {name: i for i, name in enumerate(cols)}

            # --- Edge classification loss ---
            if adj_list is not None:
                adj = adj_list[b].to(self.device)  # (p, p)
                # Build edge labels
                edge_labels = []
                for ui in range(p):
                    for vi in range(p):
                        if ui != vi:
                            edge_labels.append(int(adj[ui, vi].item()))
                edge_labels = torch.tensor(
                    edge_labels, dtype=torch.long, device=self.device
                )
                e_logits = edge_logits[b, :E_b]  # (E_b, 2)
                loss_edge = self.edge_criterion(e_logits, edge_labels)
                total_loss = total_loss + loss_edge
                n_terms += 1

            # --- Node classification loss ---
            if node_labels_list is not None and node_logits_list[b] is not None:
                node_labels_dict = node_labels_list[b]
                other_nodes = [n for n in cols if n not in ("X", "Y")]
                node_labels_tensor = torch.tensor(
                    [node_labels_dict[n] for n in other_nodes],
                    dtype=torch.long, device=self.device,
                )
                n_logits = node_logits_list[b]  # (K, 8)
                loss_node = self.node_criterion(n_logits, node_labels_tensor)
                total_loss = total_loss + loss_node
                n_terms += 1

        if n_terms > 0:
            total_loss = total_loss / n_terms

        self.log(
            f"{split}_loss", total_loss,
            on_step=(split == "train"), on_epoch=True, prog_bar=True,
            batch_size=B,
        )
        return total_loss

    def training_step(self, batch, batch_idx):
        return self._compute_loss(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._compute_loss(batch, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.lr, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.max_epochs
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "epoch"}]

### Graph Utilities (same as baseline, kept intact)

In [ ]:
def transform_proba_to_DAG(
    nodes: typing.List[str],
    pred: np.ndarray,
) -> np.ndarray:
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    G.add_edge("X", "Y")

    x_index, y_index = np.unravel_index(
        np.argsort(pred.ravel())[::-1], pred.shape
    )
    for i, j in zip(x_index, y_index):
        n1 = nodes[i]
        n2 = nodes[j]
        if i == j:
            continue
        if ((n1 == "X") and (n2 == "Y")) or ((n1 == "Y") and (n2 == "X")):
            continue
        if pred[i, j] > 0.5:
            G.add_edge(n1, n2)
            if not nx.is_directed_acyclic_graph(G):
                G.remove_edge(n1, n2)

    return nx.to_numpy_array(G)


def graph_nodes_representation(graph, nodelist):
    adjacency_matrix = nx.adjacency_matrix(
        graph, nodelist=nodelist
    ).todense()
    return tuple(adjacency_matrix.flatten())


def create_graph_label():
    graph_label = {
        nx.DiGraph([("X", "Y"), ("v", "X"), ("v", "Y")]): "Confounder",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("Y", "v")]): "Collider",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("v", "Y")]): "Mediator",
        nx.DiGraph([("X", "Y"), ("v", "X")]): "Cause of X",
        nx.DiGraph([("X", "Y"), ("v", "Y")]): "Cause of Y",
        nx.DiGraph([("X", "Y"), ("X", "v")]): "Consequence of X",
        nx.DiGraph([("X", "Y"), ("Y", "v")]): "Consequence of Y",
        nx.DiGraph({"X": ["Y"], "v": []}): "Independent",
    }
    nodelist = ["v", "X", "Y"]
    adjacency_label = {
        graph_nodes_representation(graph, nodelist): label
        for graph, label in graph_label.items()
    }
    return graph_label, adjacency_label


def get_labels(adjacency_matrix, adjacency_label):
    result = {}
    for variable in adjacency_matrix.columns.drop(["X", "Y"]):
        submatrix = adjacency_matrix.loc[
            [variable, "X", "Y"], [variable, "X", "Y"]
        ]
        key = tuple(submatrix.values.flatten())
        result[variable] = adjacency_label[key]
    return result

### Inference (node classification via the model)

In [ ]:
@torch.no_grad()
def infer_single(
    df: pd.DataFrame,
    model: ADIAModel,
    device: str = "cpu",
) -> pd.DataFrame:
    """
    Run inference on a single DataFrame.
    Returns adjacency matrix as pd.DataFrame using the node classification
    path for non-X/Y nodes and the edge classification path for X/Y edges.
    """
    model = model.eval()
    cols = list(df.columns)
    p = len(cols)
    nodes = cols

    edge_data, edge_types = build_edge_tensor(df)
    edge_data_t = torch.tensor(edge_data).unsqueeze(0).to(device)   # (1, E, 3, N)
    edge_types_t = torch.tensor(edge_types).unsqueeze(0).to(device) # (1, E)
    edge_mask_t = torch.ones(1, len(edge_types), dtype=torch.bool, device=device)

    edge_logits, node_logits_list = model(
        edge_data_t, edge_types_t, edge_mask_t, [cols]
    )

    # --- Build adjacency matrix from edge logits ---
    edge_probs = torch.softmax(edge_logits[0], dim=-1)[:, 1]  # (E,)
    E_mat = np.zeros((p, p))
    count = 0
    for i in range(p):
        for j in range(p):
            if i != j:
                E_mat[i, j] = edge_probs[count].item()
                count += 1

    adj = transform_proba_to_DAG(nodes, E_mat).astype(int)
    A = pd.DataFrame(adj, columns=nodes, index=nodes)

    # --- Override non-X/Y node classification with the dedicated head ---
    if node_logits_list[0] is not None:
        _, adjacency_label = create_graph_label()
        other_nodes = [n for n in cols if n not in ("X", "Y")]
        node_preds = torch.argmax(node_logits_list[0], dim=-1)  # (K,)
        col_idx = {name: i for i, name in enumerate(cols)}
        x_idx = col_idx.get("X")
        y_idx = col_idx.get("Y")

        for k, node_name in enumerate(other_nodes):
            pred_class = CLASS_NAMES[node_preds[k].item()]
            # Map class to subgraph pattern and update adjacency
            u_idx = col_idx[node_name]
            # Reset edges involving this node
            A.loc[node_name, :] = 0
            A.loc[:, node_name] = 0
            # Re-apply pattern
            patterns = {
                "Confounder":     [(node_name, "X"), (node_name, "Y")],
                "Collider":       [("X", node_name), ("Y", node_name)],
                "Mediator":       [("X", node_name), (node_name, "Y")],
                "Cause of X":     [(node_name, "X")],
                "Cause of Y":     [(node_name, "Y")],
                "Consequence of X": [("X", node_name)],
                "Consequence of Y": [("Y", node_name)],
                "Independent":    [],
            }
            for (src, dst) in patterns.get(pred_class, []):
                A.loc[src, dst] = 1

    return A

### CrunchDAO train / infer interface

In [ ]:
def train(
    X_train: typing.Dict[str, pd.DataFrame],
    y_train: typing.Dict[str, pd.DataFrame],
    model_directory_path: str,
) -> None:
    keys = list(X_train.keys())
    X_list = [X_train[k] for k in keys]
    y_list = [y_train[k] for k in keys]

    _, adjacency_label = create_graph_label()
    node_weights = compute_class_weights(y_list, adjacency_label)

    dataset = CausalEdgeDataset(X_list, y_list)
    loader = DataLoader(
        dataset, batch_size=4, shuffle=True,
        drop_last=True, num_workers=4,
        collate_fn=collate_fn,
    )

    wrapper = ADIAModelWrapper(
        d=D_MODEL,
        node_class_weights=node_weights,
        lr=1e-3,
        max_epochs=20,
    )

    trainer = pl.Trainer(
        accelerator="auto",
        max_epochs=20,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=True,
    )
    trainer.fit(wrapper, loader)

    model_path = os.path.join(model_directory_path, "model.pt")
    torch.save(wrapper.model.state_dict(), model_path)
    print(f"Model saved to {model_path}")


def infer(
    X_test: typing.Dict[str, pd.DataFrame],
    model_directory_path: str,
    id_column_name: str,
    prediction_column_name: str,
) -> pd.DataFrame:
    model_path = os.path.join(model_directory_path, "model.pt")
    model = ADIAModel(d=D_MODEL)
    model.load_state_dict(
        torch.load(model_path, map_location="cpu", weights_only=True)
    )
    model.eval()
    device = "cpu"  # safe default for inference

    submission = {}
    for name in tqdm(X_test, desc="Inferring"):
        df = X_test[name]
        nodes = list(df.columns)
        A = infer_single(df, model, device)
        for i in nodes:
            for j in nodes:
                submission[f"{name}_{i}_{j}"] = int(A.loc[i, j])

    submission_series = pd.Series(submission)
    submission_df = submission_series.reset_index()
    submission_df.columns = [id_column_name, prediction_column_name]
    return submission_df

### Local Evaluation (mirrors baseline notebook)

In [ ]:
def evaluate_local(
    X_train: dict,
    y_train: dict,
    test_size: float = 0.2,
    random_state: int = 42,
    max_samples: int = 200,  # reduce for quick local eval
):
    from sklearn.model_selection import train_test_split

    keys = list(X_train.keys())
    train_keys, test_keys = train_test_split(
        keys, test_size=test_size, random_state=random_state
    )
    test_keys = test_keys[:max_samples]

    # ---- Build dataset and train ----
    X_tr = [X_train[k] for k in train_keys]
    y_tr = [y_train[k] for k in train_keys]

    _, adjacency_label = create_graph_label()
    node_weights = compute_class_weights(y_tr, adjacency_label)

    dataset = CausalEdgeDataset(X_tr, y_tr)
    loader = DataLoader(
        dataset, batch_size=4, shuffle=True,
        drop_last=True, num_workers=0,
        collate_fn=collate_fn,
    )
    wrapper = ADIAModelWrapper(
        d=D_MODEL, node_class_weights=node_weights, lr=1e-3, max_epochs=10
    )
    trainer = pl.Trainer(
        accelerator="auto", max_epochs=10,
        logger=True, enable_checkpointing=False,
    )
    trainer.fit(wrapper, loader)

    # ---- Inference & scoring ----
    model = wrapper.model.eval()
    y_pred, y_true = [], []
    for name in tqdm(test_keys, desc="Evaluating"):
        df = X_train[name]
        A = infer_single(df, model, DEVICE)
        predicted = get_labels(A, adjacency_label)
        ground_truth = get_labels(y_train[name], adjacency_label)
        for k in predicted:
            y_pred.append(predicted[k])
            y_true.append(ground_truth[k])

    y_pred = pd.Series(y_pred)
    y_true = pd.Series(y_true)
    scores = {}
    for label in y_true.unique():
        scores[label] = np.mean(y_pred[y_true == label] == label)
    scores = pd.Series(scores)
    scores["Balanced Accuracy"] = scores.mean()
    print(scores)
    return scores

### Local Training & Evaluation

In [ ]:
import crunch

crunch = crunch.load_notebook()
X_train, y_train, X_test = crunch.load_data()

# Quick local eval (reduce max_samples for speed)
scores = evaluate_local(X_train, y_train, max_samples=500)
print(scores)

### CrunchDAO Test & Submit

In [ ]:
crunch.test(no_determinism_check=True)

print("Submit at: https://hub.crunchdao.com/competitions/causality-discovery/submit/via/notebook")